# 02 — Annotation Analysis and Cleaning Verification
### Project: Autonomous Warehouse AI
**Objective**: Visually verify ground-truth annotations across all 6 warehouse classes, execute data cleaning, convert polygon annotations, eliminate duplicate images, and verify train/val/test splitting.


In [ ]:
import os
import sys
import json
import glob
from PIL import Image
from IPython.display import Image as IPImage, display

sys.path.append('../src')
from clean_dataset import clean_and_harmonize
from prepare_yolo_dataset import prepare_yolo_splits


## 1. Ground Truth Visual Inspection
Verify bounding box placement on real warehouse images for each standardized class.


In [ ]:
display(IPImage(filename="../results/figures/sample_annotated_images.png"))


## 2. Execute Data Cleaning Pipeline
- Purge non-target classes (`cart`, `white_roll`).
- Deduplicate identical image files.
- Convert 267 polygon annotations into enclosing bounding boxes.
- Clamp coordinates strictly to `[0.0, 1.0]`.


In [ ]:
# Run cleaning pipeline
cleaning_stats = clean_and_harmonize(
    wh_dir="../data/warehouse_robot_raw",
    arm_dir="../data/robotic_arm_raw",
    output_dir="../data/processed/warehouse_cleaned",
    report_path="../results/metrics/dataset_cleaning_report.json"
)


## 3. Review Cleaning Report
Compare raw ingested counts vs final cleaned annotations.


In [ ]:
with open("../results/metrics/dataset_cleaning_report.json") as f:
    report = json.load(f)

print("Duplicates Removed:", report["cleaning_actions"]["duplicates_removed"])
print("Polygons Converted:", report["cleaning_actions"]["polygons_converted_to_bboxes"])
print("Cleaned Total Images:", report["cleaned_counts"]["total_images"])
print("Cleaned Total Annotations:", report["cleaned_counts"]["total_annotations"])
print("Cleaned Class Breakdown:", json.dumps(report["cleaned_counts"]["class_annotations"], indent=2))


## 4. Leak-Free Train / Val / Test Split
Create isolated 70% train, 15% val, and 15% test splits with fixed random seed (42).


In [ ]:
prepare_yolo_splits(
    cleaned_dir="../data/processed/warehouse_cleaned",
    output_standard="../data/processed/warehouse_yolo",
    output_balanced="../data/processed/warehouse_yolo_balanced"
)
